# For Run From Scratch Without Plots


In [ ]:
# -*- coding: utf-8 -*-
"""
 * Copyright (c) 2023 Alexandre Gauvain, Ronan Abhervé, Jean-Raynald de Dreuzy
 *
 * This program and the accompanying materials are made available under the
 * terms of the Eclipse Public License 2.0 which is available at
 * http://www.eclipse.org/legal/epl-2.0, or the Apache License, Version 2.0
 * which is available at https://www.apache.org/licenses/LICENSE-2.0.
 *
 * SPDX-License-Identifier: EPL-2.0 OR Apache-2.0
"""

# ---- LIBRAIRIES

# PYTHON PACKAGES
import sys
import os
import numpy as np
import pandas as pd

# ROOT DIRECTORY
from os.path import dirname, abspath
try:
    root_dir = '/home/bb/Documents/01_Git_Repository/01-HydroModPy-dev'
except NameError:
    root_dir = os.getcwd()
sys.path.append(root_dir)

# HYDROMODPY MODEULES
from hydromodpy import watershed_root

# ---- FUNCTION LAUNCH

def run_hydromodpy(watershed_name):

    # ---- PERSONAL PATHS

    example_path = os.path.join(root_dir, "examples", "11_for run from scratch without plots/")
    data_path = os.path.join(example_path, "data/")

    # The folder out_path is created in the example_path root directory:
    out_path = os.path.join(root_dir,'examples', 'results')
    # Or define it manually
    # out_path = 'C:/Simulations/HydroModPy/'

    print('The results of the example will be saved here :', out_path)

    # ---- EXTRACT CATCHMENT

    # Name of the study site
    watershed_name = watershed_name
    print('##### '+watershed_name.upper()+' #####')

    # Regional DEM
    dem_path = os.path.join(data_path, 'regional dem.tif')

    # Outlet coordinates of the catchment
    from_xyv = [327816.965, 6777886.670, 150, 10 , 'EPSG:2154']

    # Extract the catchment from a regional DEM
    BV = watershed_root.Watershed(dem_path=dem_path,
                                  out_path=out_path,
                                  load=False,
                                  watershed_name=watershed_name,
                                  from_lib=None, # os.path.join(root_dir,'watershed_library.csv')
                                  from_dem=None, # [path, cell size]
                                  from_shp=None, # [path, buffer size]
                                  from_xyv=from_xyv, # [x, y, snap distance, buffer size]
                                  bottom_path=None, # path
                                  save_object=True)

    # Paths necessary for the script
    stable_folder = os.path.join(out_path, watershed_name, 'results_stable')
    simulations_folder = os.path.join(out_path, watershed_name, 'results_simulations')

    # ---- MODEL PARAMETRIZATION

    # Name of the model/simulation
    model_name = 'Test_Galaxy_v0'

    # Import modules
    BV.add_settings()
    BV.add_climatic()
    BV.add_hydraulic()

    # Frame settings
    BV.settings.update_model_name(model_name) # Name of the model/simulation
    BV.settings.update_box_model(True)
    BV.settings.update_sink_fill(False)
    BV.settings.update_simulation_state('steady') # Transient
    BV.settings.update_check_model(plot_cross=False, check_grid=True)
    BV.settings.update_dis_perlen(dis_perlen=False)

    # Climatic settings
    BV.climatic.update_recharge(350 / 1000 / 365, sim_state=BV.settings.sim_state)
    BV.climatic.update_first_clim('mean') # or 'first or value

    # Hydraulic settings
    BV.hydraulic.update_nlay(3)
    BV.hydraulic.update_lay_decay(1) # 1 if not activated
    BV.hydraulic.update_bottom(None) # Set a value to set a flat bottom
    BV.hydraulic.update_thick(50) # Not consider if bottom != of None
    BV.hydraulic.update_hk(1e-5 * 24 * 3600) # m/d
    BV.hydraulic.update_sy(1/100) # -
    BV.hydraulic.update_hk_decay(0, min_value=None, log_transf=False) # Exponential decay with depth : 1/10 (about half decrease at 10m)
    BV.hydraulic.update_sy_decay(0, min_value=None, log_transf=False)
    BV.hydraulic.update_ss_decay(0, min_value=None, log_transf=False)
    BV.hydraulic.update_hk_vertical(None) # or [ [1e-5, [0, 20]], [1e-6, [20,80]] ]
    BV.hydraulic.update_cond_drain(None)

    # Boundary settings
    BV.settings.update_bc_sides(None, None)
    BV.add_oceanic('None')

    # Particle tracking settings
    BV.settings.update_input_particles(zone_partic = os.path.join(simulations_folder,model_name,'_postprocess/_rasters/seepage_areas_t(0).tif'),
                                       track_dir = 'backward',
                                       bore_depth = False,
                                       cell_div = 1, # 1
                                       )

    # ---- FLOW MODEL

    # Pre-processing
    model_modflow = BV.preprocessing_modflow(for_calib=False)

    # Processing
    success_modflow = BV.processing_modflow(model_modflow, write_model=True, run_model=True)

    # Post-processing
    if success_modflow == True:
        BV.postprocessing_modflow(model_modflow,
                                  watertable_elevation=True,
                                  watertable_depth=True,
                                  seepage_areas=True,
                                  outflow_drain=True,
                                  groundwater_flux=True,
                                  groundwater_storage=True,
                                  accumulation_flux=True,
                                  persistency_index=False, # only in transient
                                  intermittency_monthly=False, # only in transient
                                  intermittency_weekly=False, # only in transient
                                  intermittency_daily=False, # only in transient
                                  export_all_tif=False)

    # ---- PARTICLE TRACKING

    # Pre-processing
    if success_modflow == True:
        model_modpath = BV.preprocessing_modpath(model_modflow)

    # Processing
        success_modpath = BV.processing_modpath(model_modpath, write_model=True, run_model=True)

    # Post-processing
    if success_modpath == True:
        BV.postprocessing_modpath(model_modpath,
                                  ending_point=True,
                                  starting_point=True,
                                  pathlines_shp=False,
                                  particles_shp=False,
                                  random_id=None) # None

    # ---- POST PROCESSING

    timeseries_results = BV.postprocessing_timeseries(model_modflow=model_modflow,
                                                      model_modpath=model_modpath,
                                                      subbasin_results=True,
                                                      datetime_format=False)

    netcdf_results = BV.postprocessing_netcdf(model_modflow,
                                              datetime_format=False)

    watertable_depth_output = timeseries_results.watertable_depth

    return watertable_depth_output

# ---- RUN THE SCRIPT

if __name__ == '__main__':
    watertable_depth_output = run_hydromodpy('Example_11_Galaxy') # watershed_name

# ---- NOTES


[INFO]       __  __          __           __  ____          ________     


[INFO]      / / / /         / /          /  \/   /         / / __  /     


[INFO]     / /_/ /_  ______/ /________  /       /___  ____/ / /_/ /_  __ 


[INFO]    / __  / / / / __  / ___/ __ \/ /\,-/ / __ \/ __  / ____/ / / / 


[INFO]   / / / / /_/ / /_/ / /  / /_/ / /   / / /_/ / /_/ / /   / /_/ /  


...

writing loc particle data


FloPy is using the following executable to run the model: ../../../../../bin/linux/mp6
Processing basic data ...
Checking head file ...
Checking budget file and building index ...
 
Run particle tracking simulation ...
Processing Time Step     1 Period     1.  Time =  1.00000E+00                                                                       


Particle tracking complete. Writing endpoint file ...                                                                               
End of MODPATH simulation. Normal termination.


[INFO] Exported catchment time series to /home/bb/Documents/01_Git_Repository/01-HydroModPy-dev/examples/results/Example_11_Galaxy/results_simulations/Test_Galaxy_v0/_postprocess/_timeseries


[INFO] Exporting MODFLOW results as NetCDF for model Test_Galaxy_v0
